In [18]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
import os

In [19]:
ticker_name = "HDFCBANK.NS"
ticker = yf.Ticker(ticker_name)

if os.path.exists("hdfcbank_data.csv"):
    data = pd.read_csv("hdfcbank_data.csv", index_col=0, parse_dates=True)
    data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
    data = data.dropna()
else:
    data = ticker.history(period="6mo", interval="1d")
    data.to_csv("hdfcbank_data.csv")

S = data["Close"].iloc[-1]
print("Current Spot Price:", round(S, 2))

Current Spot Price: 810.3


/var/folders/3t/3xjt3f516dx8r11ngt2k1w_m0000gn/T/ipykernel_61983/1894222470.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv("hdfcbank_data.csv", index_col=0, parse_dates=True)


In [20]:
# Fix multi-index columns
data.columns = data.columns.get_level_values(0)

# Now compute log returns
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))
data = data.dropna()

voldaily = data["Log Return"].std()
vol = voldaily * np.sqrt(252)

print("Annualized Volatility:", vol)

Annualized Volatility: 0.22183678319988068


In [21]:
def d1(S, K, T, r, sigma):
    return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def d2(S, K, T, r, sigma):
    return d1(S, K, T, r, sigma) - sigma * np.sqrt(T)

def delta(S, K, T, r, sigma, option_type="call"):
    if option_type == "call":
        return norm.cdf(d1(S,K,T,r,sigma))
    else:
        return norm.cdf(d1(S,K,T,r,sigma)) - 1

def gamma(S, K, T, r, sigma):
    return norm.pdf(d1(S,K,T,r,sigma)) / (S * sigma * np.sqrt(T))

def vega(S, K, T, r, sigma):
    return S * norm.pdf(d1(S,K,T,r,sigma)) * np.sqrt(T)

In [22]:
S = data["Close"].iloc[-1]
r = 0.053

option_data = [
    ("28-Apr", 17, "OTM Put", 750),
    ("28-Apr", 17, "ATM Call", 810),
    ("28-Apr", 17, "ATM Put", 810),
    ("28-Apr", 17, "OTM Call", 870),

    ("26-May", 45, "OTM Put", 750),
    ("26-May", 45, "ATM Call", 810),
    ("26-May", 45, "ATM Put", 810),
    ("26-May", 45, "OTM Call", 870),
]

df = pd.DataFrame(option_data, columns=["Expiry", "Days", "Option", "Strike"])
df

,Expiry,Days,Option,Strike
0,28-Apr,17,OTM Put,750
1,28-Apr,17,ATM Call,810
2,28-Apr,17,ATM Put,810
3,28-Apr,17,OTM Call,870
4,26-May,45,OTM Put,750
5,26-May,45,ATM Call,810
6,26-May,45,ATM Put,810
7,26-May,45,OTM Call,870


In [23]:
greeks = []

for i in range(len(df)):
    K = df.loc[i, "Strike"]
    T = df.loc[i, "Days"] / 365
    opt_type = "call" if "Call" in df.loc[i, "Option"] else "put"
    
    greeks.append({
        "Delta": delta(S, K, T, r, vol, opt_type),
        "Gamma": gamma(S, K, T, r, vol),
        "Vega": vega(S, K, T, r, vol)
    })

greeks_df = pd.DataFrame(greeks)

df = pd.concat([df, greeks_df], axis=1)

df

,Expiry,Days,Option,Strike,Delta,Gamma,Vega
0,28-Apr,17,OTM Put,750,-0.045441,0.002463,16.706247
1,28-Apr,17,ATM Call,810,0.533167,0.010248,69.523174
2,28-Apr,17,ATM Put,810,-0.466833,0.010248,69.523174
3,28-Apr,17,OTM Call,870,0.079362,0.003809,25.840888
4,26-May,45,OTM Put,750,-0.132289,0.003392,60.917934
5,26-May,45,ATM Call,810,0.550763,0.006270,112.585077
6,26-May,45,ATM Put,810,-0.449237,0.006270,112.585077
7,26-May,45,OTM Call,870,0.214816,0.004627,83.091272


In [24]:
print("Spot Price:", S)
print("Volatility:", vol)

df

Spot Price: 810.2999877929688
Volatility: 0.22183678319988068


,Expiry,Days,Option,Strike,Delta,Gamma,Vega
0,28-Apr,17,OTM Put,750,-0.045441,0.002463,16.706247
1,28-Apr,17,ATM Call,810,0.533167,0.010248,69.523174
2,28-Apr,17,ATM Put,810,-0.466833,0.010248,69.523174
3,28-Apr,17,OTM Call,870,0.079362,0.003809,25.840888
4,26-May,45,OTM Put,750,-0.132289,0.003392,60.917934
5,26-May,45,ATM Call,810,0.550763,0.006270,112.585077
6,26-May,45,ATM Put,810,-0.449237,0.006270,112.585077
7,26-May,45,OTM Call,870,0.214816,0.004627,83.091272


In [25]:
file_name = f"{ticker}_greeks_analysis.xlsx"

with pd.ExcelWriter(file_name) as writer:
    
    # Main Greeks table
    df.to_excel(writer, sheet_name="Greeks", index=False)
    
    # Optional: Summary info
    summary = pd.DataFrame({
        "Metric": ["Spot Price", "Volatility", "Risk-Free Rate"],
        "Value": [S, vol, r]
    })
    
    summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"Excel file saved as: {file_name}")

Excel file saved as: yfinance.Ticker object <HDFCBANK.NS>_greeks_analysis.xlsx
